In [1]:
import pandas as pd
from rapidfuzz import process

In [2]:
file_path = r"C:\Users\nihar\Downloads\Extensive_A_Z_medicines_dataset_of_India.csv"

df = pd.read_csv(file_path)

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (256476, 24)


,id,name,price(₹),Is_discontinued,manufacturer_name,type,pack_size_label,short_composition1,short_composition2,substitute0,...,Consolidated_Side_Effects,use0,use1,use2,use3,use4,Chemical Class,Habit Forming,Therapeutic Class,Action Class
0,1,Augmentin 625 Duo Tablet,223.42,False,Glaxo SmithKline Pharmaceuticals Ltd,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg),Penciclav 500 mg/125 mg Tablet,...,"Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections,NaN,NaN,NaN,NaN,NaN,No,ANTI INFECTIVES,NaN
1,2,Azithral 500 Tablet,132.36,False,Alembic Pharmaceuticals Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),NaN,Zithrocare 500mg Tablet,...,"Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections,NaN,NaN,NaN,NaN,Macrolides,No,ANTI INFECTIVES,Macrolides
2,3,Ascoril LS Syrup,118.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml),Solvin LS Syrup,...,"Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus,NaN,NaN,NaN,NaN,NaN,No,RESPIRATORY,NaN
3,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),NaN,Lcfex Tablet,...,"Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,Treatment of Allergic conditions,NaN,NaN,NaN,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation)
4,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),NaN,Lcfex Tablet,...,"Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,Treatment of Allergic conditions,NaN,NaN,NaN,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation)


In [3]:
df.columns

Index(['id', 'name', 'price(₹)', 'Is_discontinued', 'manufacturer_name',
       'type', 'pack_size_label', 'short_composition1', 'short_composition2',
       'substitute0', 'substitute1', 'substitute2', 'substitute3',
       'substitute4', 'Consolidated_Side_Effects', 'use0', 'use1', 'use2',
       'use3', 'use4', 'Chemical Class', 'Habit Forming', 'Therapeutic Class',
       'Action Class'],
      dtype='object')

In [4]:
df = df.rename(columns={
    'name': 'brand_name',
    'price(₹)': 'price',
    'manufacturer_name': 'manufacturer',
    'short_composition1': 'comp1',
    'short_composition2': 'comp2',
    'Therapeutic Class': 'therapeutic_class'
})

In [5]:
df = df[['brand_name', 'price', 'manufacturer', 'comp1', 'comp2',
         'therapeutic_class',
         'substitute0','substitute1','substitute2','substitute3','substitute4']]

df.head()

,brand_name,price,manufacturer,comp1,comp2,therapeutic_class,substitute0,substitute1,substitute2,substitute3,substitute4
0,Augmentin 625 Duo Tablet,223.42,Glaxo SmithKline Pharmaceuticals Ltd,Amoxycillin (500mg),Clavulanic Acid (125mg),ANTI INFECTIVES,Penciclav 500 mg/125 mg Tablet,Moxikind-CV 625 Tablet,Moxiforce-CV 625 Tablet,Fightox 625 Tablet,Novamox CV 625mg Tablet
1,Azithral 500 Tablet,132.36,Alembic Pharmaceuticals Ltd,Azithromycin (500mg),NaN,ANTI INFECTIVES,Zithrocare 500mg Tablet,Azax 500 Tablet,Zady 500 Tablet,Cazithro 500mg Tablet,Trulimax 500mg Tablet
2,Ascoril LS Syrup,118.00,Glenmark Pharmaceuticals Ltd,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml),RESPIRATORY,Solvin LS Syrup,Ambrodil-LX Syrup,Zerotuss XP Syrup,Capex LS Syrup,Broxum LS Syrup
3,Allegra 120mg Tablet,218.81,Sanofi India Ltd,Fexofenadine (120mg),NaN,RESPIRATORY,Lcfex Tablet,Etofex 120mg Tablet,Nexofex 120mg Tablet,Fexise 120mg Tablet,Histafree 120 Tablet
4,Allegra 120mg Tablet,218.81,Sanofi India Ltd,Fexofenadine (120mg),NaN,RESPIRATORY,Lcfex Tablet,Etofex 120mg Tablet,F Din 120mg Tablet,Nexofex 120mg Tablet,Fexise 120mg Tablet


In [6]:
df = df.dropna(subset=['brand_name', 'comp1'])

df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

df = df.drop_duplicates()

print("After cleaning:", df.shape)

After cleaning: (256359, 11)


In [7]:
df['salt_clean'] = (
    df['comp1'].fillna('') + " " + df['comp2'].fillna('')
).str.lower()

df['salt_clean'] = df['salt_clean'].str.replace(r'[^a-z0-9 ]', '', regex=True)
df['salt_clean'] = df['salt_clean'].str.replace(r'\d+mg', '', regex=True)
df['salt_clean'] = df['salt_clean'].str.strip()

df[['comp1','comp2','salt_clean']].head()

,comp1,comp2,salt_clean
0,Amoxycillin (500mg),Clavulanic Acid (125mg),amoxycillin clavulanic acid
1,Azithromycin (500mg),NaN,azithromycin
2,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml),ambroxol 5ml levosalbutamol 5ml
3,Fexofenadine (120mg),NaN,fexofenadine
4,Fexofenadine (120mg),NaN,fexofenadine


In [8]:
df = df.sample(min(50000, len(df)), random_state=42)

In [9]:
df.to_csv("medicines_clean.csv", index=False)
print("Saved successfully")

Saved successfully


In [10]:
def fuzzy_search(name):
    choices = df['brand_name'].dropna().tolist()
    match, score, _ = process.extractOne(name, choices)
    
    if score < 60:
        return None
    
    return match

In [11]:
def affordability_score(original, cheapest):
    if original == 0:
        return 0
    savings = (original - cheapest) / original
    return round(min(savings * 10, 10), 1)

In [12]:
def get_alternatives(medicine_name):
    
    # Step 1: fuzzy match
    matched_name = fuzzy_search(medicine_name)
    
    if not matched_name:
        return {"error": "Medicine not found"}
    
    med = df[df['brand_name'] == matched_name].iloc[0]
    
    salt = med['salt_clean']
    original_price = med['price']
    
    # Step 2: find same salt
    same_salt = df[df['salt_clean'] == salt].copy()
    
    # fallback
    if same_salt.empty or len(same_salt) < 2:
        substitutes = [
            med.get('substitute0'),
            med.get('substitute1'),
            med.get('substitute2'),
            med.get('substitute3'),
            med.get('substitute4')
        ]
        
        substitutes = [s for s in substitutes if pd.notna(s)]
        
        return {
            "medicine": matched_name,
            "price": original_price,
            "note": "Using dataset substitutes",
            "alternatives": substitutes
        }
    
    # Step 3: sort
    same_salt = same_salt.sort_values(by='price')
    
    same_salt['savings_percent'] = (
        (original_price - same_salt['price']) / original_price * 100
    )
    
    # Step 4: affordability
    cheapest_price = same_salt.iloc[0]['price']
    score = affordability_score(original_price, cheapest_price)
    
    # Step 5: output
    return {
        "medicine": matched_name,
        "price": original_price,
        "therapeutic_class": med.get('therapeutic_class'),
        "affordability_score": score,
        "alternatives": same_salt[['brand_name','price','manufacturer','savings_percent']]
                        .head(5)
                        .to_dict(orient="records")
    }

In [13]:
print(get_alternatives("crocin"))

{'medicine': 'Mecocin Injection', 'price': np.float64(51.0), 'therapeutic_class': 'VITAMINS MINERALS NUTRIENTS', 'affordability_score': np.float64(8.4), 'alternatives': [{'brand_name': 'My 12 OD Injection', 'price': 8.0, 'manufacturer': 'Zeelab Pharmacy Pvt Ltd', 'savings_percent': 84.31372549019608}, {'brand_name': 'Neuroage 1500mcg Injection', 'price': 24.0, 'manufacturer': 'Allenge India', 'savings_percent': 52.94117647058824}, {'brand_name': 'Nurlin 1500mcg Injection', 'price': 25.0, 'manufacturer': 'Plenus Pharmaceuticals Pvt Ltd', 'savings_percent': 50.98039215686274}, {'brand_name': 'Multimore 1500mcg Injection', 'price': 25.0, 'manufacturer': 'Mits Healthcare Pvt Ltd', 'savings_percent': 50.98039215686274}, {'brand_name': 'Melbin 1500 Injection', 'price': 25.0, 'manufacturer': 'Meltic Healthcare Pvt Ltd', 'savings_percent': 50.98039215686274}]}


In [14]:
print(get_alternatives("augmentin 625"))

{'medicine': 'Ikclav 625 LB Tablet', 'price': np.float64(144.0), 'therapeutic_class': 'ANTI INFECTIVES', 'affordability_score': np.float64(7.6), 'alternatives': [{'brand_name': 'Ozimentin LB 500mg/125mg Tablet', 'price': 35.0, 'manufacturer': 'Oddiant Formulations', 'savings_percent': 75.69444444444444}, {'brand_name': 'Acuclav LB Dry Syrup', 'price': 45.84, 'manufacturer': 'Macleods Pharmaceuticals Pvt Ltd', 'savings_percent': 68.16666666666666}, {'brand_name': 'Moxyvive LB Dry Syrup', 'price': 70.0, 'manufacturer': 'Vrevive Medicure Pvt. Ltd.', 'savings_percent': 51.388888888888886}, {'brand_name': 'Clavonip Kid Tablet', 'price': 70.53, 'manufacturer': 'Nippon Seiyaku Pvt Ltd', 'savings_percent': 51.020833333333336}, {'brand_name': 'Moxipil CL Syrup', 'price': 72.0, 'manufacturer': 'Psychotropics India Ltd', 'savings_percent': 50.0}]}


In [15]:
print(get_alternatives("randomtest"))

{'medicine': 'Xadomet G 1mg/500mg Tablet', 'price': np.float64(52.8), 'therapeutic_class': 'ANTI DIABETIC', 'affordability_score': np.float64(8.5), 'alternatives': [{'brand_name': 'METPRIDE 2 MG/500 MG TABLET', 'price': 8.12, 'manufacturer': 'Alkem Laboratories Ltd', 'savings_percent': 84.62121212121212}, {'brand_name': 'Glynet-M1 Tablet', 'price': 10.0, 'manufacturer': 'Zeelab Pharmacy Pvt Ltd', 'savings_percent': 81.06060606060606}, {'brand_name': 'Glynet-M1 SR Forte Tablet', 'price': 20.0, 'manufacturer': 'Zeelab Pharmacy Pvt Ltd', 'savings_percent': 62.121212121212125}, {'brand_name': 'Glimison-M 0.5mg/500mg Tablet', 'price': 24.0, 'manufacturer': 'Unison Pharmaceuticals Pvt Ltd', 'savings_percent': 54.54545454545454}, {'brand_name': 'Davapride MF Forte 2mg/1000mg Tablet SR', 'price': 24.0, 'manufacturer': 'Davaindia Generic Pharmacy', 'savings_percent': 54.54545454545454}]}
